# Session 4 - RAG and Multimodal RAG
*A combined theory + hands-on notebook - follows the lecture deck slide-by-slide, with live demos where they fit.*

---

### Agenda (from the deck)
1. Introduction to RAG - why retrieval matters
2. Understanding Embeddings - turning text (and other modalities) into vectors
3. RAG workflow - Retrieval and Generation
4. Chunking strategies
5. Vector similarity search - kNN / ANN, distance metrics, indexing (HNSW)
6. Active management of vector databases - CRUD
7. RAG evaluation
8. From Search to RAG + advanced RAG techniques
9. Multimodal RAG - ViT, CLIP, the multimodal pipeline

> **Three hands-on demos** are inserted where the deck introduces the idea:
> **(1)** Chunking strategies + retrieval, **(2)** an end-to-end RAG pipeline (retrieve -> augment -> generate),
> **(3)** a minimal CLIP image-text matching demo for multimodal RAG.

> **Running this:** `Runtime -> Run all`. **CPU is fine** (free Colab). The first run downloads a few
> small models and one Wikipedia article. Generation cells (Qwen) take ~10-30s each on CPU.

## Setup - install and import
Run this **first**.

> **If you ever see `NameError: name 'torch' is not defined`**, an incompatible `transformers` was
> already loaded. Do **Kernel -> Restart** (Colab: *Runtime -> Restart session*), then **Run All**.
> The cell below pins a torch-compatible version and self-heals for you.

In [ ]:
# =============================================================================
# SETUP  -  RUN THIS CELL FIRST  (works on Google Colab and local machines)
# =============================================================================
# transformers 5.x needs PyTorch >= 2.4; on older torch it disables PyTorch and you
# get "NameError: name 'torch' is not defined". We pin transformers 4.46.3 (works on
# old AND new torch) and self-heal: if a wrong version is already loaded, we restart.
import sys, subprocess, importlib.metadata as _md

def _installed(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

# 1) pin a torch-compatible transformers into THIS kernel's Python
if not (_installed("transformers") or "").startswith("4.46"):
    print("Installing transformers==4.46.3 ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.46.3", "accelerate"], check=True)

# 2) if an incompatible transformers is already live in memory, restart once
_live = sys.modules.get("transformers")
if _live is not None and not getattr(_live, "__version__", "").startswith("4.46"):
    print(f"Incompatible transformers ({_live.__version__}) loaded - restarting kernel.")
    print(">>> When it restarts, just choose 'Run All' again. <<<")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("Could not auto-restart - restart the kernel manually, then Run All.")
    raise SystemExit

# 3) install the other libraries this notebook needs (safe; won't touch transformers)
for pkg in ["sentence-transformers", "faiss-cpu", "wikipedia", "nltk",
            "scikit-image", "pillow", "matplotlib"]:
    if _installed(pkg.replace("scikit-image", "scikit_image")) is None and \
       _installed(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# 4) imports + helpers
import warnings; warnings.filterwarnings("ignore")
import numpy as np, textwrap
import torch, transformers
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import nltk
for res in ["punkt", "punkt_tab"]:
    try:
        nltk.download(res, quiet=True)
    except Exception:
        pass

def show(text, width=95):
    """Pretty-print long text so it wraps nicely in the notebook."""
    print(textwrap.fill(str(text).strip(), width=width))

print(f"Setup OK - transformers {transformers.__version__}, torch {torch.__version__} - CPU ready.")

---
# 1 - Introduction to RAG

## What is RAG?
**Retrieval-Augmented Generation (RAG)** enhances an LLM's answers by **retrieving information from
external knowledge sources before generating a response**.

- An LLM's knowledge is **frozen at training time**.
- RAG injects **real-time or domain-specific data** at query time -> more informed, current answers,
  **without retraining** the model.

## Why retrieval matters
- **Adapts** an LLM to a private/specialised domain **without full retraining** - practical and cost-efficient.
- Keeps answers **up-to-date, grounded in facts, and relevant**.

## What LLMs don't know (the gap RAG fills)
| Gap | Why the LLM can't help alone |
|---|---|
| **Private data** | It can't see your confidential/internal documents. |
| **Real-time data** | It was trained on past data and does not update automatically. |
| **Hard-to-access info** | Some knowledge simply isn't widely available online. |

## How the RAG process works - "R-A-G"
| Step | Action | What happens |
|---|---|---|
| **R** | **Retrieve** | When you ask a question, the system searches a database / document set for information related to your query. |
| **A** | **Augment** | That retrieved information is **added to the prompt**, giving the model the context it needs. |
| **G** | **Generate** | The model uses **its training + the new context** to produce a factual, grounded answer. |

## Applications of RAG
- **Code generation** - use a codebase as the knowledge base; retrieve repo-specific classes/functions/style so generated code and developer Q&A fit the actual project.
- **Enterprise chatbots** - grounded in the company's own manuals, support guides, and FAQs; fewer generic or wrong answers.
- **Specialised domains** - legal and medical: retrieve from case files, journals, and private data for accurate, secure, precise answers.

---
# 2 - RAG workflow: Retrieval and Generation

## The Retrieval stage = Information Retrieval (IR)
**Information Retrieval** is the process of finding relevant, unstructured text from large repositories
based on a user query. Key components of an IR system:

- **Document collection** - the dataset/database of text (web pages, documents, ...).
- **Indexing** - organising the data (tokenization, stop-word removal, ...) to make search fast.
- **Query processor** - analyses the user input to understand **intent**, not just keywords.
- **Ranking algorithms** - score documents by relevance (e.g. **BM25** or vector-space models).

## The evolution of search -> RAG
- Transformer models like **BERT** sharply improved search quality (Google, Bing).
- This enabled **semantic search** - matching **meaning and intent**, not just exact keywords.
- **Text-generation** models became popular for directly answering questions...
- ...but they **hallucinate**: confidently produce incorrect, outdated, or fabricated facts.
- The fix: connect LLMs to **trusted, up-to-date external data** -> this is exactly **RAG**
  (retrieval + generation for accurate responses).

---
# 3 - Understanding Embeddings

Embeddings are the **backbone of RAG's document search**.

- Converting raw text into a computer-readable form means turning it into **numeric feature vectors**
  (arrays of numbers). This is called **text representation**.
- Good representations capture both the **linguistic information** and the **semantics** (meaning) of text.
- **Deep learning models** are the popular way to produce these representations - the vectors are called **embeddings**.

## Semantic search with embeddings
- Embeddings place text in a **vector space where similar meanings are close together**.
- **Dense retrieval** uses embeddings to find the documents nearest to a query by **semantic similarity**.
- The search process is:

  `Query -> convert to embedding -> find nearest vectors -> return relevant results`

- **Chunking is essential:** large documents are split into smaller parts *before* embedding, to improve retrieval accuracy (Section 4).

## Three model-based search categories
- **Dense retrieval** - turn query and documents into embeddings; searching becomes finding the query's **nearest neighbours**.
- **Reranking** - a second-stage model **re-scores a shortlist** of results against the query and reorders them.
- **RAG** - add an LLM at the end to *generate* an answer from the retrieved results (Section 8).

## Dense retrieval vs keyword search (e.g. BM25)
- Dense retrieval **captures meaning even when exact keywords are absent**; keyword search relies on exact word matching and often misses contextually relevant results.
- **Limitations of dense retrieval:** may return irrelevant results when no good match exists; needs similarity thresholds; struggles with exact-phrase matching; performance drops in **unseen domains**.

---
# 4 - Chunking

**Why chunk at all?** Transformer models have a **limited context window** - we can't feed them texts
longer than the number of tokens they support. So long documents must be split.

## One vector per document (weaker)
- Embed only part of the document (e.g. title/intro) - captures the gist but **ignores most content**.
- Or embed chunks and **average** them into a single vector - **averaging loses information**; the
  representation gets over-compressed and no longer reflects the document's richness.

## Multiple vectors per document (better)
- **Chunk the document into smaller pieces and embed each chunk.**
- Generally performs better: **better coverage** of the text, and each vector captures an **individual
  concept**, giving a **more expressive search index**.

## Chunking strategies (the best choice depends on your texts and queries)
- **Each sentence = a chunk** - can be too granular; vectors miss surrounding context.
- **Each paragraph = a chunk** - great for short paragraphs; otherwise ~3-8 sentences per chunk.
- **Add context** to chunks that lean on their surroundings:
  - add the **document title** to each chunk;
  - add some text **before/after** each chunk so chunks **overlap** and share surrounding context.

The demo below builds all three (**sentence**, **paragraph**, **overlapping window**) and compares them.

### Demo 1 - Chunking strategies and their effect on retrieval
**Pipeline:** load a long Wikipedia article -> build 3 chunk sets -> embed with `all-MiniLM-L6-v2` ->
index each in **FAISS** -> retrieve for a question -> answer with a fast **extractive QA** model.
We use a lightweight extractive QA model here so we can quickly compare *retrieval quality* across strategies.

In [ ]:
# Step 1 - load a long document (Wikipedia article)
import wikipedia
title = "Artificial intelligence"
document = wikipedia.page(title, auto_suggest=False).content
print(f"Loaded '{title}'  |  {len(document):,} characters")

In [ ]:
# Step 2 - define the three chunking strategies
from nltk.tokenize import sent_tokenize

def sentence_chunking(text):
    "STRATEGY A: one sentence per chunk."
    return sent_tokenize(text)

def paragraph_chunking(text):
    "STRATEGY B: one paragraph per chunk (drop very short lines)."
    return [p.strip() for p in text.split("\n") if len(p.strip()) > 50]

def overlapping_window_chunking(text, window_size=3, overlap=1, title=""):
    "STRATEGY C: sliding window of sentences that OVERLAP, with the title added as context."
    sentences = sent_tokenize(text)
    step = window_size - overlap
    chunks = []
    for i in range(0, len(sentences), step):
        window = " ".join(sentences[i:i + window_size])
        chunks.append(f"Document Title: {title}\n\n{window}")
    return chunks

sentence_chunks  = sentence_chunking(document)
paragraph_chunks = paragraph_chunking(document)
overlap_chunks   = overlapping_window_chunking(document, window_size=3, overlap=1, title=title)

print("CHUNK COUNTS")
print(f"  Sentence chunks : {len(sentence_chunks)}")
print(f"  Paragraph chunks: {len(paragraph_chunks)}")
print(f"  Overlap chunks  : {len(overlap_chunks)}")
print("\nSAMPLE OVERLAP CHUNK:\n")
show(overlap_chunks[20])

In [ ]:
# Step 3 - embed each chunk set with MiniLM and index it in FAISS
import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")   # small, CPU-friendly

def create_vector_store(chunks):
    embeddings = embedding_model.encode(chunks, show_progress_bar=False)
    index = faiss.IndexFlatL2(embeddings.shape[1])   # exact L2 nearest-neighbour index
    index.add(np.array(embeddings))
    return index

sentence_index  = create_vector_store(sentence_chunks)
paragraph_index = create_vector_store(paragraph_chunks)
overlap_index   = create_vector_store(overlap_chunks)
print("Vector stores built for all three chunking strategies.")

In [ ]:
# Step 4 - retrieval + a fast extractive QA model
from transformers import pipeline

def retrieve(query, chunks, index, top_k=2):
    q = embedding_model.encode([query], show_progress_bar=False)
    _, idxs = index.search(np.array(q), top_k)
    return [chunks[i] for i in idxs[0]]

qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

def answer_from(question, chunks, index):
    retrieved = retrieve(question, chunks, index)
    context = "\n".join(retrieved)
    ans = qa_pipeline(question=question, context=context)["answer"]
    return ans, retrieved
print("Retrieval + extractive QA ready.")

In [ ]:
# Step 5 - compare the three strategies on the same questions
questions = [
    "Who coined the term artificial intelligence?",
    "What are the early goals of AI research?",
]
strategies = [
    ("A - SENTENCE",  sentence_chunks,  sentence_index),
    ("B - PARAGRAPH", paragraph_chunks, paragraph_index),
    ("C - OVERLAP",   overlap_chunks,   overlap_index),
]

for q in questions:
    print("#" * 95)
    print("QUESTION:", q)
    print("#" * 95)
    for name, chunks, index in strategies:
        ans, retrieved = answer_from(q, chunks, index)
        print(f"\n[{name}]  answer -> {ans}")
        show("   top chunk: " + retrieved[0].replace("\n", " "))
    print()

**What to observe:** the three strategies retrieve *different* context, so the extractive answer can
change. Sentence chunks are precise but context-poor; paragraph chunks carry more context but can be noisy;
overlapping windows keep neighbouring context (and the title) together - often the best balance.

---
# 5 - Vector similarity search

Retrieval = **find the vectors nearest to the query vector**.

## Nearest-neighbour search
- Retrieve the query's **k-nearest neighbours (k-NN)** in vector space.
- **Exact k-NN** scans every vector: accurate but **O(N)** - too slow at scale.
- **Approximate Nearest Neighbour (ANN)** trades a little accuracy for large speed-ups - what real vector DBs use.

## Distance / similarity metrics
| Metric | Idea | Note |
|---|---|---|
| **Cosine similarity** | Angle between vectors | Most common for text embeddings |
| **Dot product** | Projection | Equals cosine on normalised vectors |
| **Euclidean (L2)** | Straight-line distance | Used by FAISS `IndexFlatL2` (Demo 1) |
| **Manhattan (L1)** | Sum of absolute differences | Less common |

## Indexing techniques for ANN search
- **IVF** (Inverted File) - cluster the vectors, search only the nearest clusters.
- **HNSW** (Hierarchical Navigable Small World) - graph navigation (below).
- **PQ** (Product Quantization) - compress vectors to shrink memory.
- **LSH** (Locality-Sensitive Hashing) - hash similar vectors into the same bucket.

## HNSW in depth (the default in most vector DBs)
HNSW answers ANN queries by **navigating a multi-layer graph** instead of scanning every vector.

**Structure - layered "small-world" graphs (a skip-list generalised to graphs):**
- Every vector is a **node**; edges link each node to its nearest neighbours.
- Nodes are stacked in **layers**. **Top layers are sparse** with **long-range links** (express lanes);
  **layer 0 holds every node** with dense short-range links.
- A node's highest layer is picked randomly with exponentially decaying probability, so few nodes reach the top.

**Search - greedy descent:**
1. Start at a fixed **entry point** in the **top layer**.
2. **Greedily hop** to the neighbour closest to the query.
3. When no neighbour is closer, **drop down one layer** and continue.
4. Repeat to **layer 0**, then return the best `k` nodes.

Long-range top links cross big distances fast; the dense bottom layer refines the result - giving roughly **O(log N)** search instead of **O(N)**.

**Key parameters (recall vs speed / memory):**
- **M** - neighbours stored per node. Higher -> better recall, more memory.
- **efConstruction** - candidate list size while building. Higher -> better graph, slower build.
- **efSearch** - candidate list size at query time. Higher -> better recall, slower search.

**Trade-offs:** very fast + high recall at millions of vectors; costs **more memory** (stores the graph)
and builds incrementally. Default in **FAISS, Milvus, Pinecone, Weaviate, Qdrant**.

## Relevance ranking and re-ranking
Real search is a **two-stage pipeline**: a fast **retriever** builds a shortlist, then a slower, more
accurate **re-ranker** reorders it.

- **Stage 1 - Retrieve (fast, recall-focused):** get the top-N candidates cheaply (e.g. N = 100) via
  **BM25 / TF-IDF** (keyword), **dense retrieval** (embeddings), or **Learning-to-Rank (LTR)**.
- **Stage 2 - Re-rank (slower, precision-focused):** re-score only that shortlist, keep the best few (e.g. top-5) for the LLM.

**Why two stages - bi-encoder vs cross-encoder:**
| | **Bi-encoder** (retriever) | **Cross-encoder** (re-ranker) |
|---|---|---|
| How | Encodes query and doc **separately**, compares vectors | Reads **query + doc together** in one pass |
| Speed | Very fast (embed once, reuse) | Slow (one model run per pair) |
| Accuracy | Good | **Higher** - sees word-level interactions |
| Use on | Millions of docs | A shortlist of ~10-100 |

**Re-ranking techniques (from the deck):**
- **Cross-encoder models** - pairwise query-document relevance scoring.
- **Contextual re-ranking** - using pre-trained language models.
- **Diversity-aware re-ranking** (e.g. MMR) - cut redundancy, improve coverage of the result set.

**Pipeline:** `query -> retrieve top-100 (bi-encoder / BM25) -> re-rank with cross-encoder -> top-5 -> LLM`.

---
# 6 - Active management of vector databases

A **vector database** is a specialised database built to **store, index, and search high-dimensional
vector embeddings** - numerical representations of unstructured data (text, images, audio) that capture
semantic meaning. Unlike traditional databases, it enables **fast similarity search** - finding
conceptually similar data by vector **proximity**, not exact keyword matches.

## Key aspects
- **Vector embeddings** - models convert data into dense vectors where similar items sit close together.
- **Similarity search** - uses cosine similarity / Euclidean distance to find semantically close data.
- **Core applications** - RAG, semantic search engines, recommendation systems, image/audio search.
- **Features** - manage the full data lifecycle (insert, delete, update) with **metadata filtering**.

## Role in retrieval
When a user submits a query, it is **also converted into an embedding** and compared with the stored
embeddings; the database returns the **most similar vectors** (the most relevant documents), letting AI
apps quickly find contextually relevant information in huge datasets.

## CRUD operations (applied to embeddings + their metadata)
| Op | In a vector DB |
|---|---|
| **Create** | Add new embeddings (+ metadata), generated from documents/images/other data. |
| **Read** | Retrieve vectors via similarity search for a given query. |
| **Update** | Modify embeddings/metadata when the source data changes, keeping the index accurate. |
| **Delete** | Remove embeddings/metadata when data is outdated or no longer needed. |

---
# 7 - RAG evaluation

Evaluate **two things separately**: did we **retrieve** the right documents, and did we **generate** a good answer?

## 7.1 Retrieval metrics (Information Retrieval)
You need a **text archive**, a **set of queries**, and **relevance judgements** (which docs are relevant per query).
Let **k** = the cut-off rank we look at, and **R** = the total number of relevant docs for the query.

| Metric | Formula | Measures |
|---|---|---|
| **Precision@k** | (relevant in top k) / k | Of what we returned, how much is relevant |
| **Recall@k** | (relevant in top k) / R | Of all relevant docs, how many we found |
| **MRR** | mean of 1 / (rank of first relevant) | How high the **first** correct hit sits |
| **Average Precision (AP)** | mean of Precision@k at each relevant rank | Quality of the full ranking for one query |
| **MAP** | mean of AP over all queries | Overall ranking quality (**the deck's headline metric**) |
| **nDCG** | discounted gain vs the ideal order | Ranking quality with **graded** relevance |

## 7.2 Worked example
A query has **R = 4** relevant documents in the collection. The system returns this ranked list
(`1` = relevant, `0` = not):

| Rank | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| Relevant? | 1 | 0 | 1 | 1 | 0 | 1 |

- **Precision@k:**  P@1 = 1/1 = **1.00**,  P@3 = 2/3 = **0.67**,  P@5 = 3/5 = **0.60**
- **Recall@k:**  R@3 = 2/4 = **0.50**,  R@6 = 4/4 = **1.00**
- **MRR:**  first relevant is at rank 1 -> **1/1 = 1.00**
- **Average Precision:**  average Precision@k at the relevant ranks (1, 3, 4, 6):
  `(P@1 + P@3 + P@4 + P@6) / R = (1.00 + 0.67 + 0.75 + 0.67) / 4 =` **0.77**
- **MAP** = the mean of this AP across **all** test queries.

*Reading it:* high **Precision@k** = little junk near the top; high **Recall@k** = we found most relevant
docs; high **MAP / MRR** = the relevant docs sit **near the top** - which is what matters for RAG, since the
LLM only ever sees the top few chunks.

## 7.3 Generation metrics (the answer itself)
| Metric | What it measures |
|---|---|
| **Fluency** | Is the generated text coherent and natural? |
| **Perceived utility** | How helpful and informative is the response? |
| **Citation recall** | % of statements fully supported by citations. |
| **Citation precision** | % of citations that correctly support their statements. |
| **Faithfulness** | Does the answer stay consistent with the retrieved context (no hallucination)? |

**RAGAS** automates these using an **LLM-as-a-Judge**.

---
# 8 - From Search to RAG

We turn a **search pipeline** into a **RAG system** by **adding an LLM at the end**: we present the
question **plus the top retrieved documents** to the LLM and ask it to answer **using that context**.

`Query -> Retrieve top-k chunks -> Augment the prompt with them -> LLM Generates the grounded answer`

## Advanced RAG techniques
Traditional RAG can be pushed further with smarter retrieval strategies:
- **Query rewriting** - reformulate the user's query for better retrieval.
- **Multi-query RAG** - issue several query variants and merge the results.
- **Multi-hop RAG** - chain retrievals when the answer needs several linked facts.
- **Query routing** - send the query to the right index/tool/data source.
- **Agentic RAG** - an agent decides *when* and *what* to retrieve, iteratively.

### Demo 2 - an end-to-end RAG pipeline over real Wikipedia documents
A complete **Retrieve -> Augment -> Generate** pipeline over a **multi-document knowledge base built from
real Wikipedia articles**. We first **download the articles and store them as files on disk** (a realistic
ingestion step), then **load -> chunk -> embed** (`all-MiniLM-L6-v2`) **-> index** (FAISS) **-> retrieve ->
generate** with **Qwen2.5-0.5B-Instruct**.
*(Plain code so every step is visible; frameworks like LangChain package these same steps. In production the corpus would be your own private documents - here we use Wikipedia so it is large and reproducible.)*

In [ ]:
# Step 1 - DOWNLOAD the source documents from Wikipedia and STORE them as files on disk
import os, re, time, wikipedia
wikipedia.set_rate_limiting(True)   # be polite to the API (avoids intermittent errors)

def download_wikipedia_docs(topics, folder="wiki_docs", retries=4):
    """Download each Wikipedia article and save it as a .txt file under `folder`.
    Returns the list of saved file paths. Robust to the flaky Wikipedia API:
    retries with backoff, and CACHES (a title already on disk is not re-downloaded)."""
    os.makedirs(folder, exist_ok=True)
    paths = []
    for t in topics:
        fname = re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_") + ".txt"
        path = os.path.join(folder, fname)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"  cached     '{t}' -> {path}")
            paths.append(path)
            continue
        for attempt in range(1, retries + 1):
            try:
                content = wikipedia.page(t, auto_suggest=False).content
                with open(path, "w", encoding="utf-8") as f:
                    f.write(content)
                print(f"  downloaded '{t}' -> {path}  ({len(content):,} chars)")
                paths.append(path)
                break
            except Exception as e:
                if attempt < retries:
                    time.sleep(2 * attempt)          # back off, then retry
                else:
                    print(f"  (FAILED '{t}' after {retries} tries: {e})")
        time.sleep(1)                                 # gentle gap between articles
    return paths

topics = ["Artificial intelligence", "Machine learning", "Deep learning"]
doc_paths = download_wikipedia_docs(topics)
print(f"\n{len(doc_paths)} documents stored in ./wiki_docs/")

In [ ]:
# Step 2 - LOAD the stored files from disk and chunk them (overlapping sentence windows + title)
def load_docs(paths):
    docs = {}
    for p in paths:
        title = os.path.splitext(os.path.basename(p))[0].replace("_", " ")
        with open(p, encoding="utf-8") as f:
            docs[title] = f.read()
    return docs

articles = load_docs(doc_paths)

corpus_chunks = []
for title, content in articles.items():
    corpus_chunks.extend(
        overlapping_window_chunking(content, window_size=5, overlap=1, title=title)
    )

total_chars = sum(len(v) for v in articles.values())
print(f"Loaded {len(articles)} files | {total_chars:,} characters | {len(corpus_chunks):,} chunks\n")
print("SAMPLE CHUNK:\n")
show(corpus_chunks[100])

In [ ]:
# Step 3 - EMBED all chunks (MiniLM) and store them in a FAISS vector index
print("Embedding chunks (can take ~10-40s on CPU)...")
corpus_emb = np.array(embedding_model.encode(corpus_chunks, show_progress_bar=False, batch_size=64))

corpus_index = faiss.IndexFlatL2(corpus_emb.shape[1])
corpus_index.add(corpus_emb)

def retrieve_corpus(query, top_k=3):
    q = embedding_model.encode([query], show_progress_bar=False)
    _, idxs = corpus_index.search(np.array(q), top_k)
    return [corpus_chunks[i] for i in idxs[0]]

print(f"Vector store ready with {corpus_index.ntotal:,} vectors. Quick retrieval test:")
for c in retrieve_corpus("What is deep learning?"):
    show("  - " + c.replace("\n", " ")[:160])

In [ ]:
# Step 4 - load the generator LLM (Qwen2.5-0.5B-Instruct) and build the RAG function
from transformers import AutoModelForCausalLM, AutoTokenizer

qwen_id = "Qwen/Qwen2.5-0.5B-Instruct"
qtok    = AutoTokenizer.from_pretrained(qwen_id)
qmodel  = AutoModelForCausalLM.from_pretrained(qwen_id, torch_dtype="auto"); qmodel.eval()

def ask_rag(question, top_k=3, max_new_tokens=160, show_context=True):
    retrieved = retrieve_corpus(question, top_k=top_k)          # RETRIEVE
    context = "\n\n".join(retrieved)
    messages = [                                                # AUGMENT
        {"role": "system", "content":
            "You are a helpful assistant. Use ONLY the information in the Context to answer. "
            "Do NOT use any prior knowledge and do NOT guess. If the Context does not contain "
            "the answer, reply with EXACTLY this and nothing else: "
            "Answer not found in the retrieved documents."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = qtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qtok(prompt, return_tensors="pt", truncation=True, max_length=2048)
    with torch.no_grad():                                       # GENERATE
        out = qmodel.generate(**inputs, max_new_tokens=max_new_tokens,
                              do_sample=False, pad_token_id=qtok.eos_token_id)
    answer = qtok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if show_context:
        print("Top retrieved snippet:")
        show("  " + retrieved[0].replace("\n", " ")[:220])
        print()
    return answer

print("Qwen generator loaded. RAG pipeline ready.")

In [ ]:
# Step 5 - ask questions (answerable from the corpus, plus one that is NOT)
for q in [
    "In one sentence, what is machine learning?",
    "In one sentence, what is deep learning?",
    "What are the health benefits of drinking green tea?",   # not in these AI/ML docs -> should refuse
]:
    print("=" * 95)
    print("Q:", q, "\n")
    print("ANSWER ->", ask_rag(q))
    print()

**What to observe:** the corpus is now **real, large Wikipedia documents downloaded and stored on disk**
(`./wiki_docs/`), then chunked, embedded, and indexed. Answers are **grounded in the retrieved context**, and
the last question isn't covered by these AI/ML articles, so a well-prompted RAG system should **decline**
rather than hallucinate - the *faithfulness* property from Section 7.

---
# 9 - Multimodal RAG

## What is multimodal RAG?
Multimodal RAG extends RAG to **multiple data types** - text, images, tables, audio, video.

- Unlike standard (text-only) RAG, it can **retrieve and combine information across modalities**.
- It uses **modality encoders** that convert each data type into embeddings mapped into a **shared
  embedding space** for unified retrieval.
- During ingestion it may apply **OCR** to extract text from images, preprocess structured/unstructured
  data, and store embeddings in a vector DB.
- A query in one modality (e.g. text) can retrieve information in **other** modalities (images, tables, docs).
- Retrieved multimodal evidence is handed to a generative model (e.g. **GPT-4, Gemini**) to produce a grounded answer.

## Vision Transformer (ViT)
ViT applies the Transformer idea to images: instead of splitting **text** into tokens, it **cuts an image
into patches** (horizontally and vertically) and treats those patches as the "tokens".

## Multimodal embedding models - CLIP
The best-known multimodal embedding model is **CLIP (Contrastive Language-Image Pre-training)** by OpenAI.

- CLIP learns the relationship between **images and their text descriptions** by learning a **shared
  embedding space** where images and text live together.
- **Key components:** (1) an **image encoder** (ViT or CNN), (2) a **text encoder** (Transformer),
  (3) a **shared embedding space**, (4) trained with **contrastive learning** - pull correct
  image-text pairs together, push mismatched pairs apart.
- Because of this, CLIP does **zero-shot** tasks (e.g. image classification) with no task-specific training.
- **How it works:** encode the image -> encode candidate texts -> compute **cosine similarity** between
  the image embedding and each text embedding -> the highest similarity is the best match.

## The multimodal RAG process
1. **User query** (text and/or image).
2. **Encode** text and image into embeddings (e.g. with CLIP).
3. **Shared embedding space** - map both into one common space for comparison.
4. **Multimodal fusion** - merge the text + image embeddings into a single query representation.
5. **Retrieve** the most relevant multimodal items from a vector DB.
6. **Combine** the retrieved results into a richer context.
7. **Generate** the final response with a generative model (e.g. GPT-4).

## Challenges of multimodal RAG
- **High computational cost** - multimodal systems need large resources.
- **Data alignment** - hard to align embeddings across modalities; poor alignment hurts retrieval.
- **Limited evaluation metrics** - most benchmarks are text-only.
- **Scarce high-quality multimodal datasets**, especially in specialised domains.
- **Hallucination / reliability** - conflicting signals across modalities can still mislead the model.

### Demo 3 - minimal CLIP (image <-> text similarity)
The heart of multimodal retrieval is CLIP's **shared embedding space**: we can measure how well an image
matches a piece of text with **cosine similarity**. Below we take one image and let CLIP pick the caption
that best matches it - **zero-shot**, no training. This is the building block a full multimodal retriever uses.

In [ ]:
# Minimal CLIP demo: match ONE image against several candidate captions
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import matplotlib.pyplot as plt

clip_id   = "openai/clip-vit-base-patch32"
clip      = CLIPModel.from_pretrained(clip_id)
clip_proc = CLIPProcessor.from_pretrained(clip_id)

# an offline sample photo bundled with scikit-image (a cat named "chelsea")
from skimage import data
image = Image.fromarray(data.chelsea())

candidate_captions = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a car",
    "a photo of a flower",
]

inputs = clip_proc(text=candidate_captions, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    logits = clip(**inputs).logits_per_image      # image-to-text similarity scores
probs = logits.softmax(dim=1)[0]

plt.imshow(image); plt.axis("off"); plt.title("Query image"); plt.show()

print("CLIP zero-shot match (cosine similarity -> softmax):\n")
for caption, p in sorted(zip(candidate_captions, probs.tolist()), key=lambda x: -x[1]):
    print(f"  {p*100:5.1f}%   {caption}")
print("\nCLIP picks the correct caption with no task-specific training - the same image/text")
print("similarity is what a multimodal RAG system uses to retrieve across modalities.")

---
# Recap

> **RAG** grounds an LLM by **retrieving** relevant external text, **augmenting** the prompt with it, and
> letting the model **generate** a faithful answer. The engine is **embeddings** (text -> vectors) stored in a
> **vector database** and searched by **similarity** (often ANN via **HNSW**). **Chunking** decides what gets
> embedded; **evaluation** checks both retrieval (MAP) and generation (faithfulness, citations, RAGAS).
> **Multimodal RAG** extends this to images/audio/video using shared-space encoders like **CLIP**.

### Exercises
1. **Chunking:** in Demo 1, change `window_size`/`overlap` and `top_k`. How do the retrieved chunks and answers change?
2. **Retrieval:** in Demo 2, add a new policy line, re-run, and ask a question that needs it.
3. **Faithfulness:** make Demo 2 answer a question that is *almost* in the document. Does it stay grounded or drift?
4. **Multimodal:** in Demo 3, swap in another `skimage.data` image (e.g. `data.astronaut()`, `data.coffee()`) and adjust the captions.

### References (from the deck)
- Alammar, J. and Grootendorst, M., 2024. *Hands-On Large Language Models*. O'Reilly Media.
- Chen, Kornblith, Norouzi, Hinton, 2020. *A Simple Framework for Contrastive Learning of Visual Representations.*